# STORM tutorial with a scikit-learn KNN model

This notebook shows how to wrap a third-party model in the small STORM model contract. It tries the `penguins` dataset through seaborn first and falls back to the built-in Iris dataset from scikit-learn.

## 1. Local imports

Install `scikit-learn` to run this notebook. `seaborn` and `pandas` are optional; without seaborn the tutorial uses Iris.

In [ ]:
from io import BytesIO
from pathlib import Path
import html
import sys
from tempfile import TemporaryDirectory

from IPython.display import HTML, Image, display

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "storm").is_dir():
            return candidate
    raise RuntimeError("Could not find the STORM project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
from storm import (
    DataRef,
    Dataset,
    FileArtifactStore,
    MetricRegistry,
    ModelOutput,
    ModelRegistry,
    PipelineDataLoader,
    PipelineContext,
    PipelineRunner,
    PipelineSpec,
    PipelineStep,
    RunSpec,
    StepRegistry,
    StepSpec,
    Study,
    StudySpec,
    Visualization,
    VisualizationManager,
    VisualizationRegistry,
    VisualizationRequest,
    VisualizationResult,
    VisualizationSpec,
)

try:
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.datasets import load_iris
    from sklearn.metrics import confusion_matrix
    from sklearn.model_selection import train_test_split
    from sklearn.neighbors import KNeighborsClassifier
except ImportError as error:
    raise RuntimeError(
        "This tutorial needs scikit-learn. Install it in the notebook environment."
    ) from error

## 2. Load raw data and extend STORM with pipeline steps

STORM does not own dataset loading or feature engineering. The application resolves `DataRef`, defines external `PipelineStep` classes, and registers them at runtime.

In [ ]:
data_ref = DataRef(identifier="penguins-or-iris", fingerprint="sha256:runtime-demo")


def load_raw_penguins_or_iris(reference: DataRef) -> Dataset:
    try:
        import seaborn as sns

        penguins = sns.load_dataset("penguins")
        features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
        inputs = penguins[features].to_numpy(dtype=float)
        targets = penguins["species"].to_numpy()
        metadata = {
            "source": reference.identifier,
            "dataset_name": "penguins",
            "feature_names": features,
            "morphology_schema": "penguins",
        }
    except Exception as error:
        iris = load_iris()
        inputs = iris.data.astype(float)
        targets = iris.target_names[iris.target]
        metadata = {
            "source": reference.identifier,
            "dataset_name": "iris",
            "feature_names": list(iris.feature_names),
            "morphology_schema": "iris-fallback",
            "fallback_reason": type(error).__name__,
        }
    return Dataset(inputs=inputs, targets=targets, metadata=metadata)


class FiniteFeatureFilter(PipelineStep):
    step_type = "finite_feature_filter"
    version = "1"

    def process(self, context: PipelineContext) -> PipelineContext:
        data = np.asarray(context.data, dtype=float)
        targets = np.asarray(context.targets)
        keep = np.isfinite(data).all(axis=1)
        context.data = data[keep]
        context.targets = targets[keep]
        context.metadata["rows_before_filter"] = int(len(data))
        context.metadata["rows_after_filter"] = int(keep.sum())
        return context


class MorphologyRatioFeatures(PipelineStep):
    step_type = "morphology_ratio_features"
    version = "1"

    def __init__(self, epsilon: float = 1e-9) -> None:
        self.epsilon = float(epsilon)

    def process(self, context: PipelineContext) -> PipelineContext:
        data = np.asarray(context.data, dtype=float)
        names = list(context.metadata["feature_names"])

        def ratio(name: str, numerator: str, denominator: str):
            return name, data[:, names.index(numerator)] / (data[:, names.index(denominator)] + self.epsilon)

        if context.metadata.get("morphology_schema") == "penguins":
            engineered = [
                ratio("pico_shape_index", "bill_length_mm", "bill_depth_mm"),
                ratio("pico_aleta_ratio", "bill_length_mm", "flipper_length_mm"),
                ratio("pico_cuerpo_index", "bill_length_mm", "body_mass_g"),
                ratio("aleta_cuerpo_index", "flipper_length_mm", "body_mass_g"),
            ]
        else:
            engineered = [
                ("pico_shape_index", data[:, 0] / (data[:, 1] + self.epsilon)),
                ("pico_aleta_ratio", data[:, 0] / (data[:, 2] + self.epsilon)),
                ("pico_cuerpo_index", data[:, 0] / (data[:, 3] + self.epsilon)),
                ("aleta_cuerpo_index", data[:, 2] / (data[:, 3] + self.epsilon)),
            ]

        ratio_names = [name for name, _ in engineered]
        context.data = np.column_stack([data, *[values for _, values in engineered]])
        context.metadata["feature_names"] = [*names, *ratio_names]
        context.metadata["ratio_feature_names"] = ratio_names
        return context


class StratifiedTrainTestSplit(PipelineStep):
    step_type = "stratified_train_test_split"
    version = "1"

    def __init__(self, test_size: float = 0.25, random_state: int = 11) -> None:
        self.test_size = float(test_size)
        self.random_state = int(random_state)

    def process(self, context: PipelineContext) -> PipelineContext:
        train_x, test_x, train_y, test_y = train_test_split(
            context.data,
            context.targets,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=context.targets,
        )
        context.data = {"train": train_x, "test": test_x}
        context.targets = {"train": train_y, "test": test_y}
        context.metadata["test_size"] = self.test_size
        return context


steps = StepRegistry()
steps.register(FiniteFeatureFilter)
steps.register(MorphologyRatioFeatures)
steps.register(StratifiedTrainTestSplit)

pipeline = PipelineRunner.from_spec(
    PipelineSpec(
        pipeline_id="penguin-feature-extension",
        steps=(
            StepSpec("finite_feature_filter"),
            StepSpec("morphology_ratio_features"),
            StepSpec("stratified_train_test_split", {"test_size": 0.25, "random_state": 11}),
        ),
    ),
    registry=steps,
)
data_loader = PipelineDataLoader(loader=load_raw_penguins_or_iris, runner=pipeline)
prepared_dataset = data_loader(data_ref)

prepared_dataset.metadata["dataset_name"], prepared_dataset.inputs["train"].shape, prepared_dataset.metadata["ratio_feature_names"]

## 3. Wrap KNN in the STORM model contract

A STORM model only needs `fit(inputs, targets)` and `predict(inputs) -> ModelOutput`. The adapter can hold any third-party implementation internally.

In [ ]:
import __main__ as notebook_main


class SklearnKnnModel:
    def __init__(self, config):
        self.n_neighbors = int(config.get("n_neighbors", 3))
        self.weights = str(config.get("weights", "uniform"))
        self.classifier = KNeighborsClassifier(
            n_neighbors=self.n_neighbors,
            weights=self.weights,
        )

    def fit(self, inputs, targets=None):
        if targets is None:
            raise ValueError("SklearnKnnModel requires supervised targets.")
        self.classifier.fit(inputs["train"], targets["train"])
        return self

    def predict(self, inputs):
        predictions = self.classifier.predict(inputs["test"])
        return ModelOutput(
            predictions=predictions,
            metadata={
                "model": "KNeighborsClassifier",
                "n_neighbors": self.n_neighbors,
                "weights": self.weights,
            },
        )


class TestAccuracy:
    def evaluate(self, *, dataset, output, model):
        expected = dataset.targets["test"]
        predicted = output.predictions
        return sum(p == y for p, y in zip(predicted, expected, strict=True)) / len(expected)


notebook_main.SklearnKnnModel = SklearnKnnModel

## 4. Define the serializable study

Only the run configs and dataset identity belong in the spec. The sklearn class itself is registered at runtime.

In [ ]:
models = ModelRegistry()
models.register("sklearn_knn", SklearnKnnModel)

metrics = MetricRegistry()
metrics.register("test_accuracy", TestAccuracy())

spec = StudySpec(
    study_id="sklearn-knn-notebook-demo",
    data=data_ref,
    runs=(
        RunSpec("knn-1", "sklearn_knn", {"n_neighbors": 1}, seed=11),
        RunSpec("knn-3", "sklearn_knn", {"n_neighbors": 3}, seed=11),
        RunSpec("knn-5-distance", "sklearn_knn", {"n_neighbors": 5, "weights": "distance"}, seed=11),
    ),
    metrics=("test_accuracy",),
)

spec.to_dict()

## 5. Execute and select the best KNN run

The same `Study` engine handles this external model because the adapter exposes the STORM model contract.

In [ ]:
workspace = TemporaryDirectory(prefix="storm-knn-notebook-")
artifact_root = Path(workspace.name) / "artifacts"

study = Study(
    spec,
    data_loader=data_loader,
    models=models,
    metrics=metrics,
    artifacts=FileArtifactStore(artifact_root),
)
results = study.run()

rows = [
    {
        "run_id": result.run_id,
        "test_accuracy": result.metrics["test_accuracy"],
        "model_type": result.record.model_type,
        "execution_id": result.record.execution_id,
        "features": len(prepared_dataset.metadata["feature_names"]),
    }
    for result in results
]
rows

In [ ]:
best = results.select("test_accuracy", maximize=True)
best_output = best.load_output()

best.run_id, best.metrics, best_output.metadata

## 6. Add notebook-local visualizations

These views are external STORM extensions too. They consume generic `VisualizationRequest` objects and return portable HTML or PNG images.

In [ ]:
def _figure_result(fig) -> VisualizationResult:
    buffer = BytesIO()
    fig.tight_layout()
    fig.savefig(buffer, format="png", dpi=140)
    plt.close(fig)
    return VisualizationResult(content=buffer.getvalue(), media_type="image/png")


class RatioSummaryTable(Visualization):
    visualization_type = "ratio_summary_table"

    def __init__(self, precision: int = 4) -> None:
        self.precision = int(precision)

    def render(self, request: VisualizationRequest) -> VisualizationResult:
        dataset = request.data
        names = list(dataset.metadata["feature_names"])
        ratio_names = list(dataset.metadata["ratio_feature_names"])
        matrix = np.vstack([dataset.inputs["train"], dataset.inputs["test"]])
        rows = []
        for name in ratio_names:
            values = matrix[:, names.index(name)]
            rows.append(
                "<tr>"
                f"<td>{html.escape(name)}</td>"
                f"<td>{values.mean():.{self.precision}f}</td>"
                f"<td>{values.std():.{self.precision}f}</td>"
                "</tr>"
            )
        content = (
            "<table>"
            "<thead><tr><th>feature</th><th>mean</th><th>std</th></tr></thead>"
            f"<tbody>{''.join(rows)}</tbody>"
            "</table>"
        )
        return VisualizationResult(content=content, media_type="text/html")


class RatioScatterPlot(Visualization):
    visualization_type = "ratio_scatter_plot"

    def __init__(self, x: str = "pico_cuerpo_index", y: str = "aleta_cuerpo_index") -> None:
        self.x = x
        self.y = y

    def render(self, request: VisualizationRequest) -> VisualizationResult:
        dataset = request.data
        names = list(dataset.metadata["feature_names"])
        matrix = np.vstack([dataset.inputs["train"], dataset.inputs["test"]])
        labels = np.concatenate([dataset.targets["train"], dataset.targets["test"]])
        x_values = matrix[:, names.index(self.x)]
        y_values = matrix[:, names.index(self.y)]

        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for label in sorted(set(labels)):
            mask = labels == label
            ax.scatter(x_values[mask], y_values[mask], s=28, alpha=0.75, label=str(label))
        ax.set_xlabel(self.x)
        ax.set_ylabel(self.y)
        ax.legend(title="class", fontsize=8)
        ax.grid(alpha=0.25)
        return _figure_result(fig)


class ConfusionMatrixPlot(Visualization):
    visualization_type = "confusion_matrix_plot"

    def render(self, request: VisualizationRequest) -> VisualizationResult:
        expected = list(request.metadata["expected"])
        predicted = list(request.output.predictions)
        labels = sorted(set(expected) | set(predicted))
        matrix = confusion_matrix(expected, predicted, labels=labels)

        fig, ax = plt.subplots(figsize=(5.5, 4.8))
        image = ax.imshow(matrix, cmap="Blues")
        ax.set_xticks(range(len(labels)), labels=labels, rotation=35, ha="right")
        ax.set_yticks(range(len(labels)), labels=labels)
        ax.set_xlabel("prediction")
        ax.set_ylabel("target")
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        for row in range(matrix.shape[0]):
            for column in range(matrix.shape[1]):
                ax.text(column, row, int(matrix[row, column]), ha="center", va="center")
        return _figure_result(fig)


class AccuracyBarPlot(Visualization):
    visualization_type = "accuracy_bar_plot"

    def render(self, request: VisualizationRequest) -> VisualizationResult:
        rows = list(request.data)
        labels = [row["run_id"] for row in rows]
        values = [row["test_accuracy"] for row in rows]
        fig, ax = plt.subplots(figsize=(6.2, 3.8))
        ax.bar(labels, values, color="#4C78A8")
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("test accuracy")
        ax.tick_params(axis="x", rotation=20)
        for index, value in enumerate(values):
            ax.text(index, value + 0.02, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
        return _figure_result(fig)


class PredictionPreviewTable(Visualization):
    visualization_type = "prediction_preview_table"

    def __init__(self, limit: int = 12) -> None:
        self.limit = int(limit)

    def render(self, request: VisualizationRequest) -> VisualizationResult:
        expected = list(request.metadata["expected"])
        predicted = list(request.output.predictions)
        rows = []
        for index, (target, prediction) in enumerate(zip(expected, predicted, strict=True)):
            if index >= self.limit:
                break
            rows.append(
                "<tr>"
                f"<td>{index}</td>"
                f"<td>{html.escape(str(target))}</td>"
                f"<td>{html.escape(str(prediction))}</td>"
                "</tr>"
            )
        content = (
            "<table>"
            "<thead><tr><th>row</th><th>target</th><th>prediction</th></tr></thead>"
            f"<tbody>{''.join(rows)}</tbody>"
            "</table>"
        )
        return VisualizationResult(content=content, media_type="text/html")


visualizations = VisualizationRegistry()
visualizations.register(RatioSummaryTable)
visualizations.register(RatioScatterPlot)
visualizations.register(ConfusionMatrixPlot)
visualizations.register(AccuracyBarPlot)
visualizations.register(PredictionPreviewTable)
viewer = VisualizationManager(visualizations)

ratio_view = viewer.render(
    VisualizationSpec("ratio_summary_table", {"precision": 5}),
    VisualizationRequest(data=prepared_dataset),
)
prediction_view = viewer.render(
    VisualizationSpec("prediction_preview_table", {"limit": 10}),
    VisualizationRequest(
        output=best_output,
        metadata={"expected": prepared_dataset.targets["test"]},
    ),
)
scatter_view = viewer.render(
    VisualizationSpec("ratio_scatter_plot"),
    VisualizationRequest(data=prepared_dataset),
)
confusion_view = viewer.render(
    VisualizationSpec("confusion_matrix_plot"),
    VisualizationRequest(
        output=best_output,
        metadata={"expected": prepared_dataset.targets["test"]},
    ),
)
accuracy_view = viewer.render(
    VisualizationSpec("accuracy_bar_plot"),
    VisualizationRequest(data=rows),
)

display(HTML(ratio_view.content))
display(Image(data=scatter_view.content))
display(Image(data=confusion_view.content))
display(Image(data=accuracy_view.content))
display(HTML(prediction_view.content))
ratio_view.media_type, scatter_view.media_type, confusion_view.media_type, accuracy_view.media_type, prediction_view.media_type

## 7. Clean up temporary artifacts

In [ ]:
workspace.cleanup()